In [12]:
import numpy as np
import joblib as jlb
import os
from PIL import Image 
from tqdm import tqdm
import matplotlib.pyplot as plt
import gc

# Load the data
ims = jlb.load('ims.np')
mas = jlb.load('mas.np')

print(ims.shape)
print(mas.shape)

# Create output folders
os.makedirs("cleaned/images", exist_ok=True)
os.makedirs("cleaned/masks", exist_ok=True)

def clean_array(arr):
    """Clean one image or mask safely."""
    arr = np.nan_to_num(arr, nan=0.0, posinf=255.0, neginf=0.0)
    arr = np.clip(arr, 0, 255).astype(np.uint8)
    return arr

# Process and save all images
for i in tqdm(range(len(ims)), desc="Cleaning & Saving"):
    try:
        # Clean single image + mask
        img_clean = clean_array(ims[i])
        mask_clean = clean_array(mas[i])
        
        # Convert to PIL and save
        Image.fromarray(img_clean).save(f"cleaned/images/image_{i:04d}.png")
        Image.fromarray(mask_clean).save(f"cleaned/masks/mask_{i:04d}.png")
        
        # Free memory of current items
        del img_clean, mask_clean
        gc.collect()
        
    except Exception as e:
        print(f"[WARNING] Failed at index {i}: {e}")

# Display sample images
idx = 5  # choose any index
img = Image.open(f"cleaned/images/image_{idx:04d}.png")
mask = Image.open(f"cleaned/masks/mask_{idx:04d}.png")

plt.figure(figsize=(10,5))
plt.subplot(1,2,1)
plt.imshow(img)
plt.title("Cleaned Image")
plt.axis("off")

plt.subplot(1,2,2)
plt.imshow(mask, cmap="gray")
plt.title("Cleaned Mask")
plt.axis("off")
plt.show()

# Load and process a sample batch
import glob

img_files = sorted(glob.glob("cleaned/images/*.png"))
mask_files = sorted(glob.glob("cleaned/masks/*.png"))

# Load small subset or batch-by-batch
sample_imgs = [np.array(Image.open(f)).astype(np.float32)/255.0 for f in img_files[:50]]
sample_masks = [np.array(Image.open(f)).astype(np.float32)/255.0 for f in mask_files[:50]]

print("Sample normalized image shape:", sample_imgs[0].shape)

✅ Configuration loaded
